# Block 6 — decision, entity resolution, submission files

**Reads:** Block 5 probabilities (`workspace/block5/final/scores`), Block 2 candidates, Block 1 records.
**Writes:** `workspace/block6/output/matching_results.tsv`,
`workspace/block6/output/candidate_pairs.tsv`, and reports under `workspace/block6/reports/`.
Packaging the submission ZIP is not this notebook's job — see the last stage for the one command.

What is decided, and on which data:

1. **Accept rule** (probability threshold, optionally "always accept an S1's top candidate above a
   lower bar") — fitted by 2-fold cross-fit on **CALIBRATION** (INDIA + US). The final policy is
   fitted on all of CALIBRATION.
2. **Entity resolver** — a record can belong to one S1 only. It keeps one owner per record and vetoes
   crowded buildings / template names. Its knobs are tuned on CALIBRATION with a cross-fit; for INDIA
   and US it is used only if that cross-fit is **not worse** than the plain accept rule.
3. **FRANCE** (no labels) uses the resolver preset `dropall2` (an S1 that lost a contested record
   keeps nothing from it; crowd and template vetoes off) and then empties S1s whose only accepted
   pair is weak (`p` below the accept threshold). A fixed rule, not tuned here.
4. **HOLDOUT is read once, for the report only** — it does not choose anything.

Both TSVs come from the same accepted pairs; `candidate_pairs.tsv` is Block 2's top-K_FINAL list per
test S1 (the scored set), so every match is inside its candidate list.

## Setup (identical first code cell in every notebook)

`BASE` is the folder that holds `src/` — found by walking up from the working directory, or taken
from `$BER_HOME`. Everything then follows: `DATA = BASE/data` (read-only, supplied by the grader),
`WORKSPACE = BASE/workspace` (everything this pipeline writes). `$BER_WORKSPACE` is set to `WORKSPACE`
**before any module is imported**, because `b2_common` / `b2_prod` compute their paths at import
time. `$HF_HOME` keeps model downloads inside the workspace. Nothing outside `BASE` is read or
written except temporary scratch files.

In [ ]:
# Setup — repository layout, workspace, module path
import os
import sys
import tempfile
from pathlib import Path


def find_base(start=None):
    """The repository folder that holds src/. Override with $BER_HOME."""
    if os.environ.get("BER_HOME"):
        return Path(os.environ["BER_HOME"]).expanduser().resolve()
    here = Path(start or os.getcwd()).resolve()
    for d in (here, *here.parents):
        if (d / "src").is_dir():
            return d
    raise FileNotFoundError(f"no folder containing src/ at or above {here}; set BER_HOME")


BASE = find_base()
SRC, DATA, WORKSPACE = BASE / "src", BASE / "data", BASE / "workspace"
WORKSPACE.mkdir(parents=True, exist_ok=True)
ROOT = str(WORKSPACE)

os.environ["BER_WORKSPACE"] = ROOT                       # b2_common / b2_prod read this at import time
os.environ["HF_HOME"] = str(WORKSPACE / "hf_cache")   # model downloads stay inside the workspace
WORK = tempfile.gettempdir()                          # scratch for large local-first writes
if str(SRC) not in sys.path:
    sys.path.insert(0, str(SRC))

print("BASE     :", BASE)
print("DATA     :", DATA, "" if DATA.is_dir() else "(MISSING — supplied by the grader)")
print("WORKSPACE:", WORKSPACE)

import gc
import importlib
import json
import re
import subprocess
import time

import numpy as np

def log(*a):
    print(time.strftime("[%H:%M:%S]"), *a, flush=True)


def jdump(obj, path):
    """JSON with numpy types converted, written to a tmp file in the same folder then renamed."""
    def _d(o):
        if isinstance(o, np.floating):
            return float(o)
        if isinstance(o, np.integer):
            return int(o)
        if isinstance(o, np.bool_):
            return bool(o)
        if isinstance(o, np.ndarray):
            return o.tolist()
        return str(o)
    path = str(path)
    os.makedirs(os.path.dirname(path), exist_ok=True)
    tmp = path + ".tmp"
    json.dump(obj, open(tmp, "w"), indent=1, default=_d)
    os.replace(tmp, path)


def require(*mods):
    """Fail early with one clear message when an expected package is missing."""
    missing = []
    for m in mods:
        try:
            importlib.import_module(m)
        except ImportError:
            missing.append(m)
    if missing:
        raise ImportError(f"missing packages: {missing} — pip install -r ../requirements.txt")

In [ ]:
# Stage 0 — config
B5_VERSION = "final"
CTX_VERSION = "v1"
B3_VERSION = "v1"
TRAIN_C = ["INDIA", "US"]
TEST_C = ["INDIA", "US", "FRANCE"]
FRANCE_PRESET = dict(preset="dropall2", crowd_on=False, tmpl_on=False)
FRANCE_DROP_WEAK_SINGLE = True

SUB_DIR = WORKSPACE / "block6"              # output/, reports/ and the zip land here
TEST_DIR = DATA / "test"                    # raw test_source*.tsv, for the official validator
OFFICIAL_VALIDATOR = DATA / "validate_submission.py"

In [ ]:
# Setup 2 — imports
import pandas as pd
import er_eval as ev
import b2_prod as b2p
import b5_catboost as b5
import b6_resolve as r65

ev.set_root(ROOT)
K_FINAL = int(b2p.load_merged_manifest()["k_final"])
_sel = [WORKSPACE / "block3" / B3_VERSION / "reports" / "selected_features.json",
        WORKSPACE / "block3" / B3_VERSION / "selected_features.json",
        SRC / "selected_features.json"]
B3_SEL = next((str(p) for p in _sel if p.exists()), None)
b5.configure(root=ROOT, b3_version=B3_VERSION, b3_selected=B3_SEL, ctx_version=CTX_VERSION,
             out_version=B5_VERSION, task_type="CPU")
OUT_DIR = SUB_DIR / "output"
REP_DIR = SUB_DIR / "reports"
OUT_DIR.mkdir(parents=True, exist_ok=True)
REP_DIR.mkdir(parents=True, exist_ok=True)
MATCH_HEADER = ["source1_entity_id", "matched_entity_ids"]
CAND_HEADER = ["source1_entity_id", "candidate_entity_ids"]
for sp, r, c in [("train", "CALIBRATION", c) for c in TRAIN_C] + [("train", "HOLDOUT", c) for c in TRAIN_C] + \
                [("test", "TEST", c) for c in TEST_C]:
    assert os.path.exists(b5.score_path(sp, r, c).replace(".parquet", ".done.json")), f"Block 5 scores missing: {sp}/{r}/{c}"
log(f"K_FINAL={K_FINAL} | {b5.B5_CODE_VERSION} / {r65.B6_CODE_VERSION}")

## 1 · Accept rule — cross-fit on CALIBRATION

In [ ]:
# Stage 1 — decision policy
CAL = b5.load_scores("train", "CALIBRATION", TRAIN_C)
SC_CAL = b5.scorer_for(ev, CAL)
F_CROSS, POL, POLS = b5.crossfit_decision(ev, SC_CAL, CAL, CAL["p"].to_numpy(np.float64))
log(f"CALIBRATION cross-fit S1-macro F0.5 = {float(F_CROSS.mean()):.6f} | fold policies {POLS}")
log("FINAL accept policy (fitted on all of CALIBRATION):", POL)
jdump({"policy": POL, "fold_policies": POLS, "calibration_crossfit_f05": float(F_CROSS.mean())}, REP_DIR / "policy.json")

## 2 · Entity resolver — tuned on CALIBRATION (cross-fit), used for INDIA/US only if not worse

In [ ]:
# Stage 2 — resolver knobs
CAL["acc"] = b5.policy_accept(CAL, CAL["p"].to_numpy(), POL)
CALP = r65.prepare(CAL[["s1_id", "cand_id", "source", "country", "p", "acc", "label"]], ev, "train", TRAIN_C)
TUNE = r65.tune(CALP, SC_CAL, None)
CF = TUNE["crossfit_vs_block6"]
K_IU = TUNE["best_knobs"] if CF["verdict"] != "WORSE" else None
log("resolver vs plain accept rule, CALIBRATION cross-fit:", CF)
log("INDIA/US resolver knobs:", K_IU if K_IU else "none (the resolver was worse on CALIBRATION -> plain accept rule)")
print(TUNE["table"].head(8).round(6).to_string())
jdump({"crossfit_vs_plain": CF, "knobs_india_us": K_IU, "best_full_delta": TUNE["best_full_delta"]}, REP_DIR / "resolver.json")
del CALP
gc.collect()

## 3 · HOLDOUT — read once, report only
INDIA/US HOLDOUT with the policy and knobs above. Nothing below feeds back into any choice.

In [ ]:
# Stage 3 — HOLDOUT report
HO = b5.load_scores("train", "HOLDOUT", TRAIN_C)
SC_HO = b5.scorer_for(ev, HO)
A1 = b5.policy_accept(HO, HO["p"].to_numpy(), POL)
HOP = r65.prepare(HO[["s1_id", "cand_id", "source", "country", "p", "label"]].assign(acc=A1), ev, "train", TRAIN_C,
                  p_lo_for_keys=0.05)
A2 = r65.resolve(HOP, r65.knobs("default", **K_IU), None)[0] if K_IU else A1
F1, F2 = r65.per_s1_f(SC_HO, HOP, A1), r65.per_s1_f(SC_HO, HOP, A2)
HOLD = {"accept_rule_only": float(F1.mean()), "with_resolver": float(F2.mean()),
        "resolver_vs_plain": ev.paired_bootstrap(F1, F2)}
for c in TRAIN_C:
    m = (SC_HO.u["country"] == c).to_numpy()
    HOLD[f"accept_rule_only|{c}"], HOLD[f"with_resolver|{c}"] = float(F1[m].mean()), float(F2[m].mean())
print(json.dumps(HOLD, indent=1, default=float))
jdump(HOLD, REP_DIR / "holdout_report.json")
del HO, HOP, A1, A2
gc.collect()

## 4 · TEST — decisions per country, then both TSVs
INDIA and US: accept rule (+ resolver if the CALIBRATION cross-fit allowed it). FRANCE: accept rule →
resolver preset `dropall2` → drop S1s whose only accepted pair is weak. Records never cross countries,
so countries are processed one at a time to keep memory flat. The candidate lists are built in the
same loop.

In [ ]:
# Stage 4 — decide + build lists per country
S1 = ev.read_records("test", "s1", ["entity_id", "country_norm"])
assert not S1["entity_id"].duplicated().any(), "duplicate S1 ids in the test source"
S1_ORDER = S1["entity_id"].astype(str).to_numpy()
S1C = S1.set_index("entity_id")["country_norm"].astype(str)
S1C.index = S1C.index.astype(str)
print("test S1 by country:", S1C.value_counts().to_dict())

MATCH_LONG, MATCH_LISTS, CAND_LISTS, STATS = [], [], [], []
for c in TEST_C:
    t0 = time.time()
    T = b5.load_scores("test", "TEST", [c])[["s1_id", "cand_id", "source", "country", "p"]].reset_index(drop=True)
    T["acc"] = b5.policy_accept(T, T["p"].to_numpy(), POL)
    n_plain = int(T["acc"].sum())
    if c == "FRANCE":
        T = r65.prepare(T, ev, "test", [c])
        acc, why = r65.resolve(T, r65.knobs(**FRANCE_PRESET), None)
        acc = np.asarray(acc, bool)
        if FRANCE_DROP_WEAK_SINGLE:
            d = pd.DataFrame({"s1": T["s1_id"].to_numpy(), "p": T["p"].to_numpy(), "acc": acc})
            g = d[d["acc"]].groupby("s1").agg(n=("p", "size"), pmax=("p", "max"))
            weak = g[(g["n"] == 1) & (g["pmax"] < POL["t"])].index
            acc = acc & ~T["s1_id"].isin(weak).to_numpy()
            log(f"  FRANCE weak single-accept S1s emptied: {len(weak):,}")
        print(r65.reason_counts(T, why))
    elif K_IU:
        T = r65.prepare(T, ev, "test", [c])
        acc = np.asarray(r65.resolve(T, r65.knobs("default", **K_IU), None)[0], bool)
    else:
        acc = T["acc"].to_numpy(bool)
    L = r65.pairs_to_long(T, acc)
    MATCH_LONG.append(L)
    STATS.append({"country": c, "scored_pairs": len(T), "accepted_by_rule": n_plain, "final_matches": len(L),
                  "s1_with_match": int(L["s1_id"].nunique()), "n_s1": int((S1C == c).sum())})
    del T, acc
    gc.collect()
    # candidate_pairs for this country: Block 2 rows at rank <= K_FINAL, in rank order (the scored set)
    frames = []
    for _m, df in b2p.iter_candidates("test", roles=["TEST"], countries=[c], k=K_FINAL):
        if len(df):
            frames.append(df.loc[df["rank"] <= K_FINAL, ["s1_id", "cand_id", "rank"]])
    cd = pd.concat(frames, ignore_index=True)
    assert not cd.duplicated(["s1_id", "cand_id"]).any(), "duplicate (s1_id, cand_id) in Block 2 test candidates"
    cd = cd.sort_values(["s1_id", "rank"], kind="stable")
    CAND_LISTS.append(cd.assign(s1_id=cd["s1_id"].astype(str), cand_id=cd["cand_id"].astype(str))
                        .groupby("s1_id", sort=False)["cand_id"].agg(",".join))
    STATS[-1]["candidate_pairs"] = len(cd)
    del cd, frames
    gc.collect()
    log(f"{c}: {STATS[-1]} ({time.time() - t0:.0f}s)")
MATCH_LONG = pd.concat(MATCH_LONG, ignore_index=True)
MATCH_LISTS = MATCH_LONG.groupby("s1_id")["match_id"].agg(lambda x: ",".join(sorted(set(x))))
CAND_LISTS = pd.concat(CAND_LISTS)
print(pd.DataFrame(STATS).to_string(index=False))

In [ ]:
# Stage 5 — write both TSVs, then validate against the submission rules
def write_tsv(path, header, lists):
    lists = lists.reindex(S1_ORDER).fillna("")
    path = str(path)
    tmp = path + ".tmp"
    with open(tmp, "w", encoding="utf-8", newline="\n") as fh:
        fh.write("\t".join(header) + "\n")
        for s, v in zip(S1_ORDER, lists.to_numpy()):
            fh.write(f"{s}\t{v}\n")
    os.replace(tmp, path)


MATCH_P, CAND_P = str(OUT_DIR / "matching_results.tsv"), str(OUT_DIR / "candidate_pairs.tsv")
write_tsv(MATCH_P, MATCH_HEADER, MATCH_LISTS)
write_tsv(CAND_P, CAND_HEADER, CAND_LISTS)
log(f"written {MATCH_P} ({os.path.getsize(MATCH_P) / 1e6:.1f} MB) and {CAND_P} ({os.path.getsize(CAND_P) / 1e6:.1f} MB)")


def read_lists(path, header):
    issues, lists, ids = [], {}, []
    with open(path, encoding="utf-8") as fh:
        head = fh.readline().rstrip("\n").split("\t")
        if head != header:
            issues.append(f"header {head} != {header}")
        for ln, line in enumerate(fh, start=2):
            parts = line.rstrip("\n").split("\t")
            if len(parts) != 2:
                issues.append(f"line {ln}: {len(parts)} tab-separated fields")
                continue
            s1, v = parts
            items = v.split(",") if v else []
            if any(x == "" for x in items):
                issues.append(f"line {ln}: empty id in list")
            if len(set(items)) != len(items):
                issues.append(f"line {ln}: duplicate ids in list")
            ids.append(s1)
            lists[s1] = items
    return head, ids, lists, issues


POOL_IDS = set(pd.concat([ev.read_records("test", s, ["entity_id"])["entity_id"].astype(str) for s in ("S2", "S3")]))


def official_cmd(script, match_p, cand_p, test_dir):
    """Build the organisers' validator command from its own --help, so the real option names are
    used instead of guessed ones. Returns (cmd, help_text) or (None, help_text)."""
    h = subprocess.run([sys.executable, str(script), "--help"], capture_output=True, text=True)
    help_txt = (h.stdout or "") + (h.stderr or "")
    opts = set(re.findall(r"--[A-Za-z][\w-]*", help_txt))

    def pick(*names):
        return next((n for n in names if n in opts), None)

    cmd = [sys.executable, str(script)]
    flags = [(pick("--matching", "--matching-results", "--matching_results", "--matches",
                   "--matching-file", "--predictions", "--pred"), str(match_p)),
             (pick("--candidate", "--candidates", "--candidate-pairs", "--candidate_pairs",
                   "--candidate-file"), str(cand_p)),
             (pick("--test-dir", "--test_dir", "--test-data", "--test", "--data-dir", "--data_dir",
                   "--dataset"), str(test_dir))]
    used = [(f, v) for f, v in flags if f]
    if not used:
        return None, help_txt
    for f, v in used:
        cmd += [f, v]
    return cmd, help_txt


def run_official(match_p, cand_p):
    """Run data/validate_submission.py if the grader supplied it. Never reports a pass it did not get."""
    if not OFFICIAL_VALIDATOR.exists():
        log(f"official validator not found at {OFFICIAL_VALIDATOR} - inline checks only")
        return None
    cmd, help_txt = official_cmd(OFFICIAL_VALIDATOR, match_p, cand_p, TEST_DIR)
    print("--- validate_submission.py --help ---")
    print(help_txt.strip()[:2000])
    if cmd is None:
        return {"ran": False, "reason": "could not recognise any option in --help; "
                                        "read the usage above and call the script by hand",
                "help": help_txt[:2000]}
    print("running:", " ".join(cmd))
    pr = subprocess.run(cmd, capture_output=True, text=True)
    return {"ran": True, "cmd": cmd, "returncode": pr.returncode,
            "stdout": pr.stdout[-4000:], "stderr": pr.stderr[-2000:]}


def validate_submission(match_p, cand_p, label=""):
    issues = []
    universe = pd.Index(S1_ORDER)
    L = {}
    for key, p, h in (("matching", match_p, MATCH_HEADER), ("candidates", cand_p, CAND_HEADER)):
        _, ids, lists, iss = read_lists(p, h)
        issues += [f"{key}: {x}" for x in iss[:20]]
        idx = pd.Index(ids)
        if idx.duplicated().any():
            issues.append(f"{key}: {int(idx.duplicated().sum())} duplicate source1_entity_id rows")
        if len(universe.difference(idx)):
            issues.append(f"{key}: {len(universe.difference(idx))} test S1 missing")
        if len(idx.difference(universe)):
            issues.append(f"{key}: {len(idx.difference(universe))} rows are not test S1")
        unk = {x for v in lists.values() for x in v} - POOL_IDS
        if unk:
            issues.append(f"{key}: {len(unk)} listed ids are not in test S2/S3, e.g. {list(unk)[:3]}")
        L[key] = lists
    bad = sum(1 for s, v in L["matching"].items() if not set(v) <= set(L["candidates"].get(s, [])))
    if bad:
        issues.append(f"{bad} S1 have matches outside their candidate list")
    res = {"file": label or match_p, "issues": issues, "PASS": not issues,
           "n_s1_rows": len(L["matching"]), "n_pairs": int(sum(len(v) for v in L["matching"].values())),
           "n_candidate_pairs": int(sum(len(v) for v in L["candidates"].values()))}
    official = run_official(match_p, cand_p)
    if official is not None:
        res["official"] = official
        if official.get("ran"):
            res["PASS"] = res["PASS"] and official["returncode"] == 0
    print("VALIDATION", "PASS" if res["PASS"] else "FAIL",
          {k: v for k, v in res.items() if k not in ("issues", "official")})
    for x in issues[:20]:
        print("  -", x)
    return res


VAL = validate_submission(MATCH_P, CAND_P, "regenerated")
jdump(VAL, REP_DIR / "validation_regenerated.json")
assert VAL["PASS"], "the regenerated files failed validation - do not package them"

# label-free sanity per country: accepts per S1 and share of empty rows
_long = MATCH_LONG.assign(country=MATCH_LONG["s1_id"].map(S1C))
print(pd.DataFrame({"n_s1": S1C.value_counts(), "pairs": _long["country"].value_counts(),
                    "s1_with_match": _long.groupby("country")["s1_id"].nunique()})
      .assign(accepts_per_s1=lambda d: (d["pairs"] / d["n_s1"]).round(3),
              empty_share=lambda d: (1 - d["s1_with_match"] / d["n_s1"]).round(4)).to_string())
del _long

## 5 · Outputs
The two TSVs are the deliverable. Assembling them into `<team_name>submission.zip` is a packaging step
outside this pipeline: this stage prints the file to submit, the pre-submission checks, and the single
command that builds the archive from an already-staged tree.

In [ ]:
# Stage 7 — outputs summary and pre-submission checks
README_P, REQS_P = BASE / "README.md", BASE / "requirements.txt"


def tsv_rows(p):
    with open(p, encoding="utf-8") as fh:
        return sum(1 for _ in fh) - 1


print("FILES TO SUBMIT as output/ in the archive")
for label, p in (("matching_results.tsv", MATCH_P), ("candidate_pairs.tsv", CAND_P)):
    print(f"  {label:22s} {os.path.getsize(p) / 1e6:9.1f} MB  {tsv_rows(p):>10,} rows  {p}")
print("\nreports:", REP_DIR)

EXPECTED = ["er_eval", "b2_common", "b2_prod", "b3_features", "b4_ce", "b4_ctx", "b5_catboost", "b6_resolve"]
have = {f.name for f in SRC.glob("*.py")}
missing_mod = [m for m in EXPECTED if f"{m}.py" not in have]
nbs = sorted(f.name for f in SRC.glob("*.ipynb") if not f.name.endswith("-checkpoint.ipynb"))
todo = [str(p) for p in (README_P, REQS_P) if not p.exists()]
print(f"\nsrc/: {len(have)} modules, {len(nbs)} notebooks")
if missing_mod:
    log("WARNING - modules missing from src/:", missing_mod)
if len(nbs) < 6:
    log(f"WARNING - only {len(nbs)} notebooks in src/ (expected 01 ... 06):", nbs)
if todo:
    log("WARNING - part of the submission but missing:", todo)

print("\nTo package, stage the tree and zip it (nothing under data/ or workspace/ is submitted):\n")
print("  STAGE=/path/to/<team_name>submission")
print("  mkdir -p $STAGE/output $STAGE/code/business_entity_resolution")
print(f"  cp '{MATCH_P}' $STAGE/output/matching_results.tsv")
print(f"  cp '{CAND_P}' $STAGE/output/candidate_pairs.tsv")
print(f"  cp -r '{SRC}' '{README_P}' '{REQS_P}' $STAGE/code/business_entity_resolution/")
print("  cp /path/to/Documentation_template.md $STAGE/      # organisers' template, at the archive root")
print("  (cd $STAGE && zip -r ../<team_name>submission.zip .)")